# GTEx projection into ARCHS4 CLAMP C2CP model

💡 **Environment:** `clamp-analyses`

Projects GTEx v8 RNA-seq TPM data into the ARCHS4 CLAMP C2CP model using `CLAMP::projectCLAMP`.
Genes are aligned between datasets; the result (LVs x samples) is saved to CSV for downstream k-means analysis.

## Libraries

In [1]:
library(data.table)
library(dplyr)
library(here)
library(CLAMP)

source(here("config.R"))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



## Output

In [2]:
output_dir <- here("output", "archs4")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)
projection_path <- file.path(output_dir, "gtex_archs4_C2CP_projection.csv")
projection_path

[1] "/home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/gtex_archs4_C2CP_projection.csv"

## Load ARCHS4 CLAMP C2CP model

In [3]:
archs4_model <- readRDS(here("output", "archs4", "archs4_CLAMP_C2CP.rds"))
archs4_genes <- rownames(archs4_model$Z)
cat("ARCHS4 model genes:", length(archs4_genes), "\n")
cat("ARCHS4 model samples:  ", ncol(archs4_model$B), "\n")

ARCHS4 model genes: 18423 
ARCHS4 model samples:   605614 


## Load and aggregate raw GTEx TPM

In [4]:
exprs_path  <- file.path(config$GTEx$DATASET_FOLDER, 'GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_tpm.gct.gz')
output_file <- config$GTEx$DATASET_FILE

if (!file.exists(output_file)) {
  dir.create(dirname(output_file), recursive = TRUE, showWarnings = FALSE)
  exprs_data <- read.table(exprs_path, header = TRUE, sep = "\t", skip = 2, check.names = FALSE)
  saveRDS(exprs_data, output_file)
  message("File successfully written to: ", output_file)
} else {
  message("Output file already exists. Skipping.")
}

Output file already exists. Skipping.



In [5]:
gtex <- readRDS(here(config$GTEx$DATASET_FILE))
gtex <- as.data.table(gtex)
aggregated_gtex <- gtex[, lapply(.SD, sum), by = Description, .SDcols = is.numeric]

genes    <- aggregated_gtex$Description
samples  <- colnames(aggregated_gtex[, -1])
data_mat <- as.matrix(aggregated_gtex[, -1])
rownames(data_mat) <- genes
colnames(data_mat) <- samples

cat("GTEx expression:", nrow(data_mat), "genes x", ncol(data_mat), "samples\n")

GTEx expression: 54592 genes x 17382 samples


## Align to ARCHS4 gene space

In [6]:
common_genes <- intersect(archs4_genes, genes)
cat("Overlapping genes:", length(common_genes), "/", length(archs4_genes), "ARCHS4 genes",
    sprintf("(%.1f%%)\n", 100 * length(common_genes) / length(archs4_genes)))

# Subset data to common genes
gtex_aligned <- data_mat[common_genes, , drop = FALSE]

# Filter model to significant LVs (AUC > 0.7, FDR < 0.05)
archs4_model_summary <- archs4_model$summary
archs4_model_summary %>%
  dplyr::filter(AUC > 0.7) %>%
  dplyr::filter(FDR < 0.01) %>%
  dplyr::pull(LV) %>%
  unique() -> significant_LVs

cat("Significant LVs:", length(significant_LVs), "\n")

# Subset model Z to common genes AND significant LVs
# (projectCLAMP requires nrow(Z) == nrow(newdata))
archs4_model_sub   <- archs4_model
archs4_model_sub$Z <- as.matrix(archs4_model$Z[common_genes, significant_LVs, drop = FALSE])

Overlapping genes: 16619 / 18423 ARCHS4 genes (90.2%)


Significant LVs: 96 


## Project with `CLAMP::projectCLAMP`

In [7]:
proj <- projectCLAMP(
  CLAMPres = archs4_model_sub,
  newdata  = gtex_aligned,
)

write.csv(as.data.frame(proj), projection_path, row.names = TRUE)
message("Projection saved to: ", projection_path)

Projection saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/gtex_archs4_C2CP_projection.csv



In [8]:
proj_check <- read.csv(projection_path, row.names = 1, check.names = FALSE)
cat("Projection dimensions:", nrow(proj_check), "LVs x", ncol(proj_check), "samples\n")
proj_check[1:5, 1:5]

Projection dimensions: 96 LVs x 17382 samples


,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
LV2,-6.15379,-1.814750,-5.055878,-6.160825,0.7425644
LV4,-12.43890,-25.602459,-15.007495,-12.781571,-31.0297362
LV6,65.84023,12.109093,60.622496,87.819814,29.7662160
LV10,25.50158,-1.716753,24.335741,24.988439,-19.2935974
LV16,14.31143,-1.826006,2.895074,15.943082,-1.9852480


## Randomized-Z projection (null baseline)

Row-permute Z so gene–pathway associations are broken, preserving the loading value distribution.

In [9]:
set.seed(42)

rand_projection_path <- file.path(output_dir, "gtex_archs4_C2CP_randomZ_projection.csv")

Z_real   <- as.matrix(archs4_model_sub$Z)
perm_idx <- sample(nrow(Z_real))
Z_rand   <- Z_real[perm_idx, , drop = FALSE]
rownames(Z_rand) <- rownames(Z_real)

archs4_model_rand   <- archs4_model_sub
archs4_model_rand$Z <- Z_rand

proj_rand <- projectCLAMP(
  CLAMPres = archs4_model_rand,
  newdata  = gtex_aligned
)

write.csv(as.data.frame(proj_rand), rand_projection_path, row.names = TRUE)
message("Randomized-Z projection saved to: ", rand_projection_path)

Randomized-Z projection saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/gtex_archs4_C2CP_randomZ_projection.csv

